<p style="text-align: right; margin: 0; line-height: 0;"><img width="100px" height="auto" src="../../.admin/img/eth_logo.png"></p>

<h1 style="text-align: center;">04 Mobile Algorithms</h1>
<hr>


## **1. Control and Coordination in Mobile Systems**

<div class="alert alert-light">

  **Name:** (add your name here)
  
  **Matriculation Number:** (add your number here)
  
</div>


### 1.1 Introduction

Control theory is about modeling how a system evolves, deciding how to influence it, and ensuring it behaves predictably and stably. Control and coordination in mobile and distributed systems can be intuitively introduced through an example that is both familiar and safety-critical: the insulin pump. The pump continuously measures blood glucose levels and adjusts insulin delivery in response. It is a clear case of a feedback control system: one that must sense, decide, and act based on incomplete and delayed information, all while ensuring stability and safety. The insulin pump is computational software embedded in a network of sensors, actuators, and algorithms. It does not simply follow a pre-programmed schedule but reacts to changing conditions in the body, much like mobile devices and distributed systems respond to changes in their environments. Each measurement can be noisy, each control action delayed, and yet the system must remain stable and converge to the desired glucose level. Understanding such a system means understanding feedback: how a device can regulate a physical process using data that reflects its own previous actions. When we generalize this to a collection of devices (each with sensors, processors, and communication links) we arrive at the idea of distributed control. Here, decisions emerge from cooperation and information exchange rather than a single central authority. This perspective leads naturally to the study of stability, convergence, and optimality in networked systems. It also highlights the practical importance of communication, since every feedback signal must pass through a channel that may be noisy, lossy, or delayed. Whether we consider an insulin pump communicating with a continuous glucose monitor, a swarm of drones coordinating through wireless links, or a mobile app adapting its sensing rate to conserve energy, the same principles apply.

This notebook introduces the **proportional (P) controller**: its correction is proportional to the current error. We use it to study feedback, stability, and the limits of P control. [Notebook 04.2: PID Controller](04.2_PID_Controller.ipynb) builds on this by adding integral and derivative action. By examining the insulin pump as a guiding example, we learn how feedback can be used to produce reliable behavior in uncertain environments. This understanding forms the basis for designing and analyzing control strategies in all types of mobile and distributed systems.

### 1.2 Linear control systems with and without feedback

Feedback occurs when two or more dynamic systems interact so that each influences the other. When this interaction forms a closed cycle, the system is called closed-loop; when it is broken, the system operates open-loop. Control is the systematic use of feedback and algorithms to influence system behavior. Every feedback loop involves three essential processes: sensing, computation, and actuation. Sensing means measuring the system’s state, such as blood glucose. Computation means comparing the measured value with the desired target and deciding on corrective action. Actuation means applying an input to influence the system, for example by injecting insulin. The goal of control design is to make the closed-loop system stable, meaning that bounded disturbances should produce bounded errors, and to achieve the desired performance, such as quick response to changes and strong rejection of disturbances. Two important advantages of feedback are robustness to uncertainty and the ability to shape the dynamics of a system. Feedback can maintain the desired operation even when system parameters vary, and it can make a system faster, smoother, or more stable. Poorly designed feedback can cause instability, amplify noise, or create unwanted couplings between subsystems.

The artificial pancreas is a modern example of feedback control, where sensing, computation, and actuation work together in a loop that maintains stability and safety in the presence of continuous biological and environmental disturbances. The figure below illustrates how feedback control applies to blood glucose regulation. At the top, the closed-loop artificial pancreas automatically adjusts insulin delivery. A glucose sensor, or continuous glucose monitor (CGM), continuously measures blood glucose levels and sends this data to a controller algorithm. The controller compares the measured glucose with the target setpoint and computes the insulin dose required to minimize the error. This insulin is delivered through a pump and cannula into the body, which changes the blood glucose concentration through the body’s natural metabolism. The updated glucose level is then measured again, closing the feedback loop. External disturbances such as meals, exercise, and stress constantly affect the system, but feedback control compensates for these effects automatically. In contrast, the open-loop system, shown at the bottom of the figure, relies on manual user input. The user measures glucose, estimates the required insulin based on experience or tables, and delivers insulin manually through the pump. This system has no automatic correction; errors can persist longer, and control depends entirely on the user’s actions.


<figure style="border: 1px solid #888; padding: 16px; margin: 16px 0; text-align: center;">
<img src="../../.admin/notebooks/picture/insulin01.png" 
     alt="Artificial Pancreas" 
     width="90%" 
     style="filter: grayscale(60%);">
     <figcaption style="margin-top: 12px; text-align: justify;"><strong>Closed loop and open loop control of insulin delivery. The proportional (P) controller is introduced below; proportional-integral-derivative (PID) control is introduced in <a href="04.2_PID_Controller.ipynb">Notebook 04.2</a>.</strong></figcaption>
</figure>



An **open-loop controller** selects its input without using the measured result to correct its action. A **closed-loop controller** measures the result, compares it with a target, and updates the input from the resulting error. Closed-loop control is widely used for practical regulation tasks because it can compensate for disturbances and uncertainty. Open-loop control can be sufficient when the system is predictable, disturbances are small, or measuring the output is impractical. The appropriate choice depends on the application; there is no single prevalence claim that applies to every kind of control system.




#### 1.2.1 Artificial Pancreas: Meals, sport, and hypoglycemia rescue


The following example illustrates the effect of closed-loop control in an artificial pancreas system. The simulation models glucose dynamics during typical daily events such as meals (which raise glucose) and physical activity (which lowers glucose). The closed-loop simulation combines a proportional (P) insulin controller with a separate glucagon-like rescue input. Together, these reduce the deviations caused by meals and sport compared with basal-only delivery. The plotted state is glucose deviation from the target in arbitrary units: zero is the target, and negative values mean below target, not a negative glucose concentration. This model does not define clinical hypoglycemia thresholds or establish that hypoglycemia is prevented, as this is only a simplified model: Meals provide temporary glucose absorption over 60 minutes; exercise lowers glucose only during the shaded sport interval. Background glucose supply balances basal insulin at the target before any events.

> 👩‍⚕️ "Hypoglycemia" means low blood glucose (sugar) levels, measured in parts per volume. It occurs when the body’s available glucose falls below what is needed for normal brain and organ function. It might happen if too much insulin is delivered, a meal is missed, or intense physical activity consumes excess glucose. The controller must prevent hypoglycemia by reducing insulin delivery when glucose levels drop too low.


<figure style="border: 1px solid #888; padding: 16px; margin: 16px 0; text-align: center;">
<img src="../../.admin/notebooks/picture/artificial_pancreas_proportional.png" alt="Open-loop and proportional closed-loop glucose responses to meals and sport, with noisy sensor measurements" style="display: block; width: 100%; max-width: 1200px; height: auto; margin: 0 auto;">
<figcaption style="margin-top: 12px; text-align: justify;"><strong>Proportional feedback with noisy glucose measurements: glucose responses to meals and sport with basal-only delivery and P control plus glucagon-like rescue.</strong></figcaption>
</figure>


In the graph above, the open-loop pump delivers constant basal insulin. A matching background glucose supply keeps the glucose deviation at zero before the first meal. Both meals have the same input: amplitude 0.8, an immediate onset, and a gradual fade over a finite 60-minute absorption window. They therefore supply the same total glucose, although their visible responses differ because the glucose level at each meal is different. After absorption ends, glucose relaxes toward its baseline.

During sport (120–160 minutes), additional glucose consumption lowers the open-loop curve. This extra consumption stops at minute 160, so glucose stops falling and begins a slow recovery toward baseline. With a 200-minute recovery time constant, less than 10% of the glucose deficit is recovered before meal 2, so the curve stays close to its low level. Meal 2 begins at minute 180 with an idealized rapid onset of glucose absorption, making the upward slope steeper, followed by a gradual fade over its finite absorption window. Glucose remains continuous at both event boundaries; its slope changes abruptly.

The closed-loop system adjusts insulin delivery proportionally to the current glucose deviation using noisy measurements and also includes a separate glucagon-like rescue input when glucose is below the target. The rescue input is a separate mechanism, not an integral or derivative term. Its recovery therefore reflects both proportional feedback and the extra rescue input, which the basal-only open-loop system does not have.

> 👩‍⚕️ In our context, **basal** means *background* or *steady*. The basal insulin rate is the small, continuous amount of insulin that an insulin pump delivers throughout the day and night to cover the body’s normal metabolic needs when no food is being eaten. It keeps glucose levels stable between meals and during sleep by balancing the liver’s slow release of glucose into the bloodstream. In contrast, **bolus insulin** refers to larger, temporary doses given at specific times, such as before meals, to handle the extra glucose coming from food.

#### 1.2.2 Proportional control with feedback and measurement

Let $x_t$ be the **state**, representing the body's internal glucose level at time $t$, and let $u_t$ be the **control input**, the amount of insulin delivered by the pump. The linear system model includes an external disturbance $d_t$, which affects the state independently of the controller:

$$
x_{t+1} = a x_t + b u_t + d_t.
$$

The parameter $a$ describes how much of the current state carries over to the next time step, and $b$ describes the effect of the control input on the state. A sensor measures the state with gain $c$, producing the **measured output** $y_t$:

$$
y_t = c\,x_t.
$$

The **proportional controller** compares the measurement with the target $x^*$ and applies the **proportional gain** $k$ (also written $K_p$) to the current error. It has no accumulated-error or derivative term:

$$
e_t = x^* - y_t, \qquad u_t = k e_t = k(x^* - c\,x_t).
$$

Substituting the control law into the system model gives

$$
x_{t+1} = a x_t + b k(x^* - c\,x_t) + d_t,
$$

so the closed-loop dynamics are

$$
\boxed{x_{t+1} = (a-bkc)\,x_t + bk\,x^* + d_t, \qquad y_t = c\,x_t.}
$$

**Direct state feedback is the special case $c=1$**, assuming perfect measurement. Then $y_t=x_t$, the error is $e_t=x^*-x_t$, and the dynamics reduce to $x_{t+1}=(a-bk)x_t+bkx^*+d_t$.

<figure style="border: 1px solid #888; padding: 16px; margin: 16px 0;">
<img src="../../.admin/notebooks/picture/feedback_parameters_with_disturbance.svg" alt="Feedback loop showing x(t), x(t+1), a, b, c, disturbance d(t), and highlighted controller gain k" style="display: block; max-width: 100%; height: auto; margin: 0 auto;">

<figcaption style="margin-top: 12px; text-align: justify;"><strong>Feedback through a sensor. The sensor gain c converts the state into the measured output y(t) = c x(t), and k sets the strength of the reaction to e(t) = x* − y(t). Parameters a and b describe state persistence and the input effect. The disturbance d(t) enters the state update independently of the controller; the one-step delay z⁻¹ makes the computed next state available as the current state at the following step.</strong></figcaption>
</figure>

<div class="alert alert-block alert-info">

**What does $z^{-1}$ mean?** This block represents a **delay of one time step**. The system calculates the next state, $x(t+1)$. The delay stores that value until the next step, when it becomes the current state, $x(t)$, used in the next calculation and in the feedback measurement. For example, with a sampling interval of one second, the delay is one second. It represents the timing of the discrete model, rather than an additional sensor or communication delay.

</div>

The diagram uses the same discrete-time notation as the equations: x(t) is the current state and x(t+1) is the next state. The system combines the retained state a x(t), the input effect b u(t), and the external disturbance d(t). In Section 1.2.3, this disturbance is zero before step 18 and -0.2 from then onward. The delayed state returns to the system update and passes through the sensor gain **c** to produce y(t). The controller compares y(t) with the target and multiplies the error by **k**. The combination $a-bkc$ determines whether the response settles or grows.

Sensor noise can also be included as $y_t=c\,x_t+\varepsilon_t$, where $\varepsilon_t$ represents measurement noise and quantization effects. A common simplified assumption is independent additive white Gaussian noise (AWGN). With this measurement, the same controller gives $x_{t+1}=(a-bkc)x_t+bkx^*+d_t-bk\varepsilon_t$.

> 👩‍⚕️ In modern models, glucose–insulin regulation involves multiple interacting variables. The state $x_t$ may include blood glucose, insulin in plasma, and insulin in tissues; the control input $u_t$ is the insulin infusion rate; and the measured output $y_t$ is the glucose concentration provided by the continuous glucose monitor (CGM). The system model then becomes
>
>$$
>x_{t+1} = A x_t + B u_t \quad \text{and} \quad y_t = C x_t
>$$
>
>Here, $A$ captures how the internal metabolic states evolve over time, $B$ describes how insulin delivery influences them, and $C$ specifies which internal quantities are actually
>measured - for example, the glucose sensor measures only blood glucose, not tissue insulin absorption or hepatic glucose production.


#### 1.2.3 Stability of proportional control

Stability is a fundamental concept in control theory because it determines whether a system behaves predictably or spirals out of control. A stable system reacts to disturbances, noise, or small errors by eventually settling back to a desired operating point. An unstable system, on the other hand, amplifies these deviations over time: Its output grows without bound, oscillates uncontrollably, or collapses entirely. In practical terms, stability means that a feedback loop works as intended by correcting errors instead of making them worse. A system is stable if it does not diverge when disturbed, and instead it returns to equilibrium.

For the model with sensor gain $c$, stability requires $|a-bkc|<1$. The examples below use direct state feedback ($c=1$), so $x_{t+1} = (a - b k) x_t + \text{constant}$ is stable if $|a - b k| < 1$. This inequality captures the mathematical idea of control. If feedback is too weak (small $k$), the system is slow to respond. If feedback is too strong (large $k$), the system overshoots or oscillates. And finally, if $|a - b k| > 1$, the system diverges and is unstable.

Use the purple slider-controlled curve to explore weak, well-tuned, and overly aggressive gains. A moderate gain gives fast convergence; an excessive gain makes the corrections overshoot and the oscillations grow. This visualizes the stability condition $|a-bk|<1$.

* For small k: $a - b k ≈ 1$ → very slow decay, stable but sluggish.
* For moderate k: $a - b k ≈ 0.25$ → fast, well-behaved convergence.
* For large k: $a - b k < -1$ → sign alternates and magnitude grows → unstable oscillation.

**Choosing the slider gain.** Here, $a=1$, $b=0.5$, and $c=1$, so stability requires $0<k<4$. The gain $k=2$ gives the fastest settling of the noise-free response: without a disturbance or sensor noise, it reaches the target in one update. With zero-mean sensor noise, this describes the average response; individual samples fluctuate. Gains below 2 respond without alternating around the equilibrium; gains between 2 and 4 produce decaying oscillations. At 4 the oscillations persist, and above 4 they grow.

At time step 18, a constant opposing disturbance of -0.2 is added to every subsequent state update. Use the slider to observe whether the controller still reaches the target.

The sensor measures $y_t=x_t+\varepsilon_t$, with Gaussian noise of standard deviation 0.03. All slider settings use the same noise samples for a fair comparison. The plot shows the true state: its small fluctuations arise because the controller reacts to noisy measurements.


In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
from io import BytesIO

# Proportional controller: y_t = x_t + epsilon_t; u_t = k*(x_set - y_t)
# Discrete-time model: x_{t+1} = a*x_t + b*u_t + d_t
a = 1.0
b = 0.5
x0 = 0.0

# Time steps
steps = 30
t = np.arange(steps)

# Desired (setpoint) value
x_set_final = 1.0     # target value after jump
setpoint_time = 10     # time step when target changes
disturbance_time = 18  # a persistent opposing load begins here
disturbance = -0.2

# Small additive sensor noise; reuse the same samples for every gain comparison.
sensor_noise_std = 0.03
sensor_noise = np.random.default_rng(7).normal(0.0, sensor_noise_std, steps)

def simulate_and_plot(k_slider):
    # Build a PNG for the existing image widget; emit no plot display output.
    with plt.ioff():
        fig = plt.figure(figsize=(12,4.5))

    # Slider-adjusted curve
    x = np.zeros(steps)
    x[0] = x0
    for i in range(1, steps):
        x_set = x_set_final if i >= setpoint_time else 0.0
        y_meas = x[i-1] + sensor_noise[i-1]
        u = k_slider * (x_set - y_meas)
        d = disturbance if i >= disturbance_time else 0.0
        x[i] = a * x[i-1] + b * u + d
    plt.plot(t, x, marker="o", color="tab:purple", label=f"Interactive gain k = {k_slider:.2f}")

    # Desired trajectory
    x_set_series = [x_set_final if i >= setpoint_time else 0.0 for i in range(steps)]
    plt.plot(t, x_set_series, 'k--', linewidth=1.5, label="Desired value (setpoint)")

    plt.axvline(disturbance_time, color="gray", linestyle=":", label="Constant disturbance begins")
    plt.axhline(0, color="gray", linestyle="--", linewidth=1)
    plt.xlabel("time step t")
    plt.ylabel("state xₜ")
    plt.title("Proportional control with sensor noise and a constant disturbance")
    plt.gca().text(
        0.02, 0.97,
        f"a = {a:.2f}, b = {b:.2f}; sensor noise std = {sensor_noise_std:.2f}\n"
        f"Slider: a - bk = {a:.2f} - {b:.2f} × {k_slider:.2f} = {a - b*k_slider:.2f}",
        transform=plt.gca().transAxes, va="top", fontsize=10,
        bbox=dict(boxstyle="round,pad=0.4", facecolor="white", edgecolor="gray", alpha=0.9)
    )
    plt.ylim(0.0, 2.0)  # Fixed vertical scale for every slider value.
    plt.legend()
    plt.tight_layout()
    try:
        with BytesIO() as buffer:
            fig.savefig(buffer, format="png", dpi=100)
            p_control_image.value = buffer.getvalue()
    finally:
        plt.close(fig)

# Update one image widget instead of appending figures to notebook output.
# Disconnect the previous observer and close old widgets when rerunning this cell.
if "p_gain_slider" in globals() and "p_gain_changed" in globals():
    p_gain_slider.unobserve(p_gain_changed, names="value")
for widget_name in ("p_control_ui", "p_control_plot", "p_control_image", "p_gain_slider"):
    old_widget = globals().get(widget_name)
    if old_widget is not None:
        old_widget.close()

p_gain_slider = widgets.FloatSlider(
    value=1.0, min=0.0, max=5.0, step=0.02,
    description='k (slider)', continuous_update=False
)
p_control_image = widgets.Image(format="png")
p_control_image.layout.max_width = "100%"

def p_gain_changed(change):
    simulate_and_plot(change["new"])

p_gain_slider.observe(p_gain_changed, names="value")
simulate_and_plot(p_gain_slider.value)
p_control_ui = widgets.VBox([p_gain_slider, p_control_image])
display(p_control_ui)


With $k=2$, the average response settles fastest, but after the disturbance it **fluctuates around 0.8 instead of the target 1.0**. Sensor noise causes the small fluctuations; the constant disturbance causes the persistent offset. The P controller needs a remaining error to produce the input that balances the constant disturbance. Increasing the gain reduces this offset, but can cause oscillations and eventually instability; $k=2$ is optimal for the noise-free settling speed, not for the smallest offset or noise sensitivity.


----
### 1.6 Hands-On Exercises

<div class="alert alert-block alert-info">Fill in your responses and save this notebook. See the <a href="../../03_Assignments/Submission-Guidelines.md">submission guidelines</a> for how to submit your work.</div>




**What is the difference between an open-loop and a closed-loop control system?**

Answer:



**Try the interactive slider in the graph above. Which gain gives the fastest settling, and does it still reach the target after the disturbance?**

Answer:



**Give one example of an application of control systems in mobile devices or everyday life. In this example, what represents the state and the control input?**

Answer:



**Why is closed-loop control useful for maintaining a target value, and when might open-loop control be sufficient?**

Answer:



**How can a closed-loop system recover from disturbances or changes in the environment?**

Answer:



**OPTIONAL: Did you read through and understand all the tasks in this Notebook? Are there any shortcomings, missing parts, or sections that should be improved?**

Answer: